# 04 · Self-refinement, scoring, and best-of-N

Companion notebook for [Chapter 5](../chapters/ch05-self-refinement.md).

**You will:** score answers three ways, run a critique-and-revise loop, and check
the counter-intuitive result that refinement helps strong models and hurts weak
ones.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device

device = get_device()
print("torch", torch.__version__)

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, QWEN_CONFIG_06_B, Qwen3Tokenizer, download_qwen3_small,
)

download_qwen3_small(kind="base", out_dir="qwen3")     # 1.5 GB, cached after the first run

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=Path("qwen3") / "tokenizer-base.json"
)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(Path("qwen3") / "qwen3-0.6B-base.pth"))
model.to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 1 · The heuristic scorer — no model call

Read the defaults as a specification: a boxed answer is worth twice a bare number,
and rambling is penalised exponentially.

In [ ]:
from reasoning_from_scratch.ch05 import heuristic_score

candidates = [
    r"The answer is \boxed{83}.",
    r"The answer is 83.",
    "The answer is eighty-three, and here is a very long explanation " * 20,
    "I do not know.",
]
for c in candidates:
    print(f"{heuristic_score(c):.3f}  {c[:60]!r}")

**Notice:** the long rambling answer with no box scores lower than the short
unboxed one, because the brevity term decays with length. This is a crude proxy
for instruction-following, and in the published results it beats average
log-probability on reasoning traces.

## 2 · Log-probability scoring

"How confident was the model in what it said" — but only over the answer tokens,
not the prompt.

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch05 import avg_logprob_answer

raw = "Half the value of $3x-9$ is $x+37$. What is the value of $x$?"
prompt = render_prompt(raw)

for answer in [r"\boxed{83}", r"We get \boxed{38}"]:
    lp = avg_logprob_answer(model, tokenizer, prompt, answer, device=device)
    print(f"{answer!r:25} avg logprob = {float(lp):.3f}")

**The higher log-probability answer is not necessarily the correct one.** A model
can be confidently wrong, and long reasoning traces make this worse — which is why
the logprob scorer underperforms the free heuristic on the reasoning model
(48.4% vs 57.8% in the published table).

## 3 · The refinement loop

In [ ]:
from reasoning_from_scratch.ch05 import self_refinement_loop, heuristic_score

raw = "Half the value of $3x-9$ is $x+37$. What is the value of $x$?"

out = self_refinement_loop(
    model, tokenizer, raw, device,
    iterations=1,
    score_fn=heuristic_score,     # TRY: None, then compare
    verbose=True,
)
print(out)

**Predict before you run with `score_fn=None`:** does the revision improve, stay the
same, or get worse?

With `score_fn=heuristic_score` the loop can always keep the draft, so it cannot be
worse than the draft *by that metric*. With `None` you always accept the revision,
which is a gamble — and on a 0.6B base model, a losing one.

## 4 · Look at the critique

The interesting artifact is not the answer, it is the critique. Read one.

In [ ]:
from reasoning_from_scratch.ch05 import make_critique_prompt, make_refine_prompt
from reasoning_from_scratch.ch03 import render_prompt

raw = "A train travels 120 km in 1.5 hours. What is its average speed in m/s?"
draft = "The average speed is 80 m/s."

print(make_critique_prompt(raw, draft))
print("=" * 70)
print(make_refine_prompt(raw, draft, "[critique would go here]"))

**The draft is wrong** (120 km / 1.5 h = 80 km/h ≈ 22.2 m/s). A good critique
catches the unit error. Watch whether the model does — and whether the revision
fixes it or introduces something new.

## 5 · The published results, for comparison

| Method | Base | Reasoning |
| --- | ---: | ---: |
| Baseline | 15.2% | 48.2% |
| Refinement, no scorer, 1 iter | 25.0% | 56.6% |
| Refinement, heuristic, 1 iter | 21.6% | **57.8%** |
| Refinement, logprob, 1 iter | 21.4% | 48.4% |

Self-improvement is a capability *multiplier*, not a substitute. It needs a model
that can already tell a good answer from a bad one.

## 6 · Exercises

1. Set `brevity_bonus` to 100 and 2000. Which direction does accuracy move, and
   why?
2. Run refinement with 3 iterations on a question the model gets right first time.
   Does it survive?
3. Build a scorer that only checks for `\boxed{}`. Compare with the full heuristic
   on ten questions.